# 03 - Ant Colony Optimization


## Idea general

Ant Colony Optimization, o ACO, es una metaheuristica poblacional inspirada en el
comportamiento de colonias de hormigas.

La idea no es copiar literalmente a las hormigas, sino usar una analogia util:
varios agentes construyen soluciones, dejan informacion sobre las partes que
usaron y esa informacion influye en las soluciones futuras.

En ACO, esa informacion se llama feromona. Las partes de una solucion que han
aparecido en buenas soluciones reciben mas feromona y se vuelven mas atractivas
para las siguientes hormigas.

Como metaheuristica, ACO no garantiza encontrar el optimo global. Su objetivo es
guiar la busqueda combinando experiencia acumulada y decisiones probabilisticas.


## Soluciones constructivas

ACO funciona especialmente bien en problemas donde una solucion se puede
construir paso a paso.

En rutas, una hormiga puede construir un tour eligiendo la siguiente ciudad. En
un camino sobre un grafo, puede elegir el siguiente arco. En asignacion, puede
ir asignando elementos a posiciones o recursos.

La idea tecnica es que una solucion completa esta formada por componentes.

Un componente puede ser una arista, una ciudad elegida, una asignacion parcial o
una decision que se agrega a la solucion.

ACO no parte necesariamente con una solucion completa y la modifica, como Tabu
Search. En cambio, muchas hormigas construyen soluciones desde cero o desde un
estado inicial.


## Feromona

La feromona representa memoria colectiva.

Si cierto componente aparece muchas veces en buenas soluciones, su nivel de
feromona aumenta. Luego, cuando una nueva hormiga debe elegir entre varios
componentes posibles, los componentes con mas feromona tienen mayor probabilidad
de ser elegidos.

Esta memoria no pertenece a una hormiga individual. Pertenece al sistema
completo. Por eso ACO se considera una metaheuristica poblacional o de colonia.

La feromona permite explotacion: aprovechar informacion aprendida de soluciones
anteriores.


## Heuristica local

Ademas de la feromona, ACO suele usar informacion heuristica del problema.

La heuristica local mide que tan atractivo parece un componente antes de conocer
toda la solucion. En TSP, por ejemplo, una ciudad cercana puede ser mas atractiva
que una ciudad lejana.

La feromona mira la experiencia acumulada. La heuristica mira una conveniencia
local inmediata.

La decision de una hormiga combina ambas fuentes de informacion. Por eso ACO no
es puramente greedy ni puramente aleatorio.


## Regla probabilistica

Cuando una hormiga debe elegir el siguiente componente, no siempre toma el mejor
de forma deterministica.

Cada componente recibe un peso segun su feromona y su heuristica. Luego la
hormiga elige probabilisticamente: los componentes con mayor peso tienen mayor
probabilidad, pero otros componentes todavia pueden ser elegidos.

El parametro `alpha` controla cuanto pesa la feromona. Si es alto, la colonia
sigue mas fuerte lo que ya aprendio.

El parametro `beta` controla cuanto pesa la heuristica local. Si es alto, las
hormigas se comportan de forma mas greedy.

Esta regla es importante porque mantiene un equilibrio. La colonia aprovecha
informacion acumulada, pero conserva exploracion al no elegir siempre lo mismo.


## Evaporacion

La evaporacion reduce la feromona con el tiempo.

Sin evaporacion, las primeras buenas decisiones podrian dominar para siempre,
incluso si no eran realmente las mejores. La evaporacion permite olvidar parte
de la informacion antigua y seguir explorando.

El parametro `rho` controla la tasa de evaporacion. Si `rho` es alto, la memoria
se pierde mas rapido. Si es bajo, la colonia conserva mas fuerte la informacion
acumulada.

La evaporacion es una parte central del metodo porque evita la convergencia
prematura.


## Deposito

Despues de construir soluciones, las hormigas depositan feromona sobre los
componentes que usaron.

La cantidad depositada depende de la calidad de la solucion. En minimizacion, una
solucion de menor costo suele depositar mas feromona. En maximizacion, una
solucion de mayor valor deberia depositar mas.

Asi la colonia refuerza patrones utiles sin imponerlos de manera absoluta.

La combinacion de deposito y evaporacion crea una memoria dinamica: se premian
buenas decisiones, pero tambien se permite que la informacion cambie.


## Estructura en Python

La forma tecnica de implementar ACO de manera general es separar el motor de la
colonia del problema especifico.

En el notebook, la estructura se implementa con estas piezas:

- `initial_state`: crea el estado inicial de una hormiga.
- `available_components(state)`: entrega los componentes que una hormiga puede
  elegir desde ese estado.
- `add_component(state, component)`: agrega una decision y produce un nuevo
  estado.
- `solution_from_state(state)`: transforma el estado final en una solucion.
- `objective(solution)`: mide la calidad de la solucion.
- `components_of(solution)`: indica que componentes reciben feromona.
- `heuristic(state, component)`: mide la conveniencia local de un componente.
- `pheromone_key(state, component)`: identifica donde se guarda la feromona.

Esta separacion hace que ACO no dependa de TSP. Lo importante es que el problema
permita construir soluciones por componentes.


In [1]:
from collections import defaultdict
from dataclasses import dataclass
from math import inf
import random

@dataclass
class AntColonyResult:
    best_solution: object
    best_value: float
    history: list
    pheromone: dict
    iterations: int


def weighted_choice(items, weights, rng):
    total = sum(weights)
    if total <= 0:
        return rng.choice(list(items))

    threshold = rng.random() * total
    cumulative = 0.0
    for item, weight in zip(items, weights):
        cumulative += weight
        if cumulative >= threshold:
            return item

    return items[-1]


def ant_colony_optimization(
    initial_state,
    available_components,
    add_component,
    solution_from_state,
    objective,
    components_of,
    *,
    heuristic=None,
    pheromone_key=None,
    ants=20,
    iterations=100,
    alpha=1.0,
    beta=2.0,
    rho=0.4,
    q=1.0,
    sense="min",
    deposit=None,
    initial_pheromone=1.0,
    seed=None,
):
    """
    Ant Colony Optimization generico para problemas constructivos.

    La solucion se construye agregando componentes. El problema define que
    componentes existen, como se agregan y como se evalua la solucion final.
    """
    if sense not in {"min", "max"}:
        raise ValueError("sense debe ser 'min' o 'max'")

    rng = random.Random(seed)
    sign = 1 if sense == "min" else -1

    if heuristic is None:
        heuristic = lambda state, component: 1.0

    if pheromone_key is None:
        pheromone_key = lambda state, component: component

    if deposit is None:
        if sense == "min":
            deposit = lambda value: q / (abs(value) + 1e-12)
        else:
            deposit = lambda value: q * max(0.0, value)

    pheromone = defaultdict(lambda: float(initial_pheromone))
    best_solution = None
    best_value = None
    best_key = inf
    history = []

    for it in range(iterations):
        ant_solutions = []

        for _ in range(ants):
            state = initial_state()

            while True:
                candidates = list(available_components(state))
                if not candidates:
                    break

                weights = []
                for component in candidates:
                    key = pheromone_key(state, component)
                    tau = pheromone[key]
                    eta = max(0.0, float(heuristic(state, component)))
                    weights.append((tau ** alpha) * (eta ** beta))

                component = weighted_choice(candidates, weights, rng)
                state = add_component(state, component)

            solution = solution_from_state(state)
            value = objective(solution)
            ant_solutions.append((solution, value))

            value_key = sign * value
            if value_key < best_key:
                best_solution = solution
                best_value = value
                best_key = value_key

        for key in list(pheromone.keys()):
            pheromone[key] *= (1.0 - rho)

        for solution, value in ant_solutions:
            amount = float(deposit(value))
            if amount <= 0:
                continue

            for key in components_of(solution):
                pheromone[key] += amount

        history.append(best_value)

    return AntColonyResult(
        best_solution=best_solution,
        best_value=best_value,
        history=history,
        pheromone=dict(pheromone),
        iterations=iterations,
    )

## Lectura del codigo

El codigo implementa un motor general de Ant Colony Optimization.

La clase `AntColonyResult` ordena la salida del algoritmo: mejor solucion,
mejor valor, historial de mejora, feromonas finales e iteraciones realizadas.

La funcion `ant_colony_optimization` recibe funciones del problema. Con ellas
puede construir soluciones sin saber si los componentes son ciudades, aristas,
tareas o asignaciones.

La variable `pheromone` guarda la memoria colectiva. Cada clave representa un
componente o decision, y su valor indica cuanta feromona tiene.

En cada iteracion se generan varias hormigas. Cada hormiga parte desde un estado
inicial y va agregando componentes mientras existan decisiones disponibles.

Para elegir un componente, el codigo calcula un peso combinando feromona y
heuristica. Luego usa una eleccion aleatoria ponderada. Esto significa que el
mejor componente no siempre se elige, pero tiene mas probabilidad.

Cuando todas las hormigas construyen sus soluciones, el algoritmo evalua la
funcion objetivo y actualiza la mejor solucion encontrada.

Despues ocurre la evaporacion: todas las feromonas conocidas se reducen. Luego
viene el deposito: las soluciones construidas refuerzan los componentes que
usaron, segun su calidad.

La idea importante es que la colonia aprende sin que exista una hormiga central.
El aprendizaje queda guardado en la feromona compartida.


## Ejemplo 1 - TSP

TSP es un ejemplo natural para ACO porque una ruta se construye ciudad por ciudad.

El estado es la ruta parcial, el componente es la siguiente ciudad y la feromona se asocia a la arista usada para avanzar desde una ciudad a otra.


In [2]:
import math

coords = [
    (0.10, 0.20), (0.25, 0.85), (0.50, 0.55),
    (0.80, 0.75), (0.90, 0.15), (0.45, 0.10),
    (0.15, 0.55), (0.65, 0.25),
]

n = len(coords)

def distance(a, b):
    ax, ay = coords[a]
    bx, by = coords[b]
    return math.hypot(ax - bx, ay - by)

def tour_length(tour):
    return sum(distance(tour[i], tour[(i + 1) % len(tour)]) for i in range(len(tour)))

def initial_state_tsp():
    return (0,)

def available_tsp(state):
    visited = set(state)
    return [city for city in range(n) if city not in visited]

def add_city(state, city):
    return state + (city,)

def solution_from_tsp_state(state):
    return state

def heuristic_tsp(state, city):
    return 1.0 / (distance(state[-1], city) + 1e-12)

def pheromone_key_tsp(state, city):
    return (state[-1], city)

def components_of_tour(tour):
    return [(tour[i], tour[i + 1]) for i in range(len(tour) - 1)]

result = ant_colony_optimization(
    initial_state_tsp,
    available_tsp,
    add_city,
    solution_from_tsp_state,
    tour_length,
    components_of_tour,
    heuristic=heuristic_tsp,
    pheromone_key=pheromone_key_tsp,
    ants=30,
    iterations=80,
    alpha=1.0,
    beta=3.0,
    rho=0.35,
    q=1.0,
    sense="min",
    seed=7,
)

print("Mejor tour:", result.best_solution)
print("Distancia:", round(result.best_value, 4))
print("Mejor valor al inicio:", round(result.history[0], 4))
print("Mejor valor final:", round(result.history[-1], 4))
print("Componentes con feromona:", len(result.pheromone))

Mejor tour: (0, 5, 7, 4, 3, 2, 1, 6)
Distancia: 2.9124
Mejor valor al inicio: 3.0289
Mejor valor final: 2.9124
Componentes con feromona: 49


## Lectura del ejemplo TSP

Cada hormiga empieza en una ciudad fija y agrega ciudades no visitadas. Para elegir la siguiente ciudad combina feromona acumulada y una heuristica local basada en distancia.

Despues de construir los tours, las aristas usadas por mejores rutas reciben mas feromona. La evaporacion evita que una arista domine demasiado pronto.


## Ejemplo 2 - Camino mas corto

El camino mas corto en un grafo es especialmente natural para ACO, porque las hormigas pueden caminar desde un origen hasta un destino eligiendo aristas.

La feromona se deposita sobre aristas que aparecen en caminos construidos. Los caminos mas cortos depositan mas feromona.


In [3]:
graph = {
    "A": {"B": 2, "C": 5, "D": 9},
    "B": {"C": 1, "E": 4},
    "C": {"D": 2, "E": 2, "F": 8},
    "D": {"F": 3},
    "E": {"F": 2},
    "F": {},
}
start, target = "A", "F"

def path_cost(path):
    if path[-1] != target:
        return 10_000
    return sum(graph[path[i]][path[i + 1]] for i in range(len(path) - 1))

def initial_path():
    return (start,)

def available_edges(path):
    last = path[-1]
    if last == target:
        return []
    visited = set(path)
    return [node for node in graph[last] if node not in visited]

def add_node(path, node):
    return path + (node,)

def solution_from_path_state(path):
    return path

def edge_heuristic(path, node):
    return 1.0 / graph[path[-1]][node]

def edge_key(path, node):
    return (path[-1], node)

def components_of_path(path):
    return [(path[i], path[i + 1]) for i in range(len(path) - 1)]

result = ant_colony_optimization(
    initial_path,
    available_edges,
    add_node,
    solution_from_path_state,
    path_cost,
    components_of_path,
    heuristic=edge_heuristic,
    pheromone_key=edge_key,
    ants=25,
    iterations=60,
    alpha=1.0,
    beta=2.0,
    rho=0.4,
    q=10.0,
    sense="min",
    seed=3,
)

print("Mejor camino:", result.best_solution)
print("Costo:", result.best_value)
print("Historial final:", result.history[-5:])

Mejor camino: ('A', 'B', 'C', 'E', 'F')
Costo: 7
Historial final: [7, 7, 7, 7, 7]


## Lectura del ejemplo Camino mas corto

El estado es el camino parcial. Los componentes disponibles son los vecinos no visitados del ultimo nodo.

Este ejemplo muestra la intuicion original de ACO: muchas hormigas prueban caminos, los caminos buenos se refuerzan y la evaporacion mantiene abierta la exploracion.


## Parametros

Los parametros mas importantes son `alpha`, `beta`, `rho`, el numero de hormigas
y el numero de iteraciones.

`alpha` controla la influencia de la feromona. `beta` controla la influencia de
la heuristica local. `rho` controla cuanto se olvida en cada evaporacion.

Muchas hormigas entregan mas exploracion por iteracion, pero tambien aumentan el
costo computacional. Muchas iteraciones permiten mas aprendizaje, pero tambien
pueden producir estancamiento si la feromona se concentra demasiado.


## Complejidad

El costo depende de cuantas hormigas se usan, cuantos pasos necesita cada
solucion y cuantos componentes candidatos se revisan en cada paso.

En problemas de rutas, una regla aproximada es que cada iteracion puede ser
costosa porque cada hormiga construye una solucion completa y en cada decision
debe comparar candidatos.

Por eso ACO puede ser mas caro que una busqueda local simple, pero a cambio
explora muchas soluciones en paralelo y acumula aprendizaje colectivo.


## Cuando se usa

ACO se usa especialmente en problemas combinatorios constructivos.

Es natural en rutas, caminos en grafos, redes, secuenciacion y asignacion,
siempre que tenga sentido construir una solucion eligiendo componentes paso a
paso.

Funciona bien cuando hay informacion local util y cuando reforzar componentes de
buenas soluciones ayuda a guiar busquedas futuras.


## Resumen

Ant Colony Optimization es una metaheuristica poblacional donde muchas hormigas
construyen soluciones usando feromona e informacion heuristica.

La feromona representa memoria colectiva. La heuristica representa conveniencia
local. La evaporacion evita que la colonia se quede pegada demasiado pronto.

Su fortaleza es que combina exploracion probabilistica con aprendizaje
colectivo. Su debilidad es que depende bastante de parametros y puede estancarse
si la feromona se concentra demasiado rapido.
